In [0]:
# =============================================================================
# 17_build_thresholds_table.py
# =============================================================================
# CADENCE:   One-off / manual re-run when source CSV changes
# PURPOSE:   Load flood threshold definitions from the restricted volume into
#            Delta, making them queryable alongside the rest of the pipeline.
# SOURCE:    config.THRESHOLDS_CSV, by default
#            /Volumes/prd_dash_lab/flood_forecasting_restricted/shared_external_volume/thresholds_Feb.csv
# OUTPUT:    config.TBL_EA_THRESHOLDS, by default
#            prd_dash_lab.flood_forecasting_restricted.ea_thresholds_all
#
# NOTES:
#   - Raw source is preserved in full; deduplication happens downstream.
#   - Schema is inferred from the CSV on first load; overwriteSchema is enabled
#     so re-runs survive column order changes in the source file.
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

import config

CSV_PATH     = config.THRESHOLDS_CSV
OUTPUT_TABLE = config.TBL_EA_THRESHOLDS


In [0]:
# =============================================================================
# 1. Load source CSV
# =============================================================================

# Every column is read as text (the file is large and mixes numbers with NULL and blanks; later steps convert what they need).
# Read as pandas first -- the CSV lives in a Volume, not a Hive path,
# so Spark cannot read it directly via spark.read.csv.
pdf = pd.read_csv(CSV_PATH, encoding="latin-1", dtype=str, keep_default_na=False, na_values=[""])

print(f"Source rows loaded: {len(pdf)}")
print(f"Columns: {pdf.columns.tolist()}")

In [0]:
# =============================================================================
# 2. Convert to Spark and write to Delta
# =============================================================================

# Create without explicit schema then let Delta infer types.
# This is safer than passing a schema when the source is externally managed
# and column types may shift between CSV exports.
df = spark.createDataFrame(pdf)

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

row_count = spark.table(OUTPUT_TABLE).count()
print(f"Written to {OUTPUT_TABLE}: {row_count} rows")


In [0]:
# =============================================================================
# 3. Verify
# =============================================================================

display(spark.table(OUTPUT_TABLE).limit(10))

dbutils.notebook.exit("success")